In [33]:
import pandas as pd
import requests
import json

df=pd.read_json("JSON_NASA2.json", lines=True)

In [34]:
df2=df.copy()
df2=df2[["geometry","properties","parameters"]]

In [35]:
df2.head(1)

,geometry,properties,parameters
0,"{'type': 'Point', 'coordinates': [-93.718, 33....","{'parameter': {'T2M': {'20150101': 2.26, '2015...","{'T2M': {'units': 'C', 'longname': 'Temperatur..."


In [36]:
#Le but de cette cellule est de nettoyer la colonne "geometry" pour obtenir 1 colonne longitude et 1 colonne latititude
# J'extraie les coordonnées dans deux colonnes puis je supprime la colonne d'origine (on n'en a plus besoin)
df2["longitude"] = df2["geometry"].apply(lambda fonction: fonction["coordinates"][0])
df2["latitude"] = df2["geometry"].apply(lambda fonction: fonction["coordinates"][1])
df2.drop(columns="geometry",inplace=True)


In [37]:
df2.head(1)
df2.shape

(72, 4)

In [41]:
#Le but de cette cellule est de "nettoyer" la colonne "properties" pour créer une colonne par paramètre et une ligne par date

df3=df2.copy()

#J'extraie le dictionnaire du dictionnaire principal parameter de chaque ligne, pour mieux naviguer
df3["parametres"] = df3["properties"].apply(lambda fonction: fonction["parameter"])
df3.drop(columns="properties",inplace=True)

#Je transforme chaque parametre en colonne distincte
df3_bis=df3["parametres"].apply(pd.Series)
df3=pd.concat([df3.drop(columns="parametres"),df3_bis],axis=1)

#Je transforme chaque item (paire date:mesure) du dictionnaire en une liste de tuples...
def convertir_en_liste(dico):
    return list(dico.items())

df3.head()
parametres=["T2M","T2M_MAX","T2M_MIN","PRECTOTCORR","RH2M","WS2M","ALLSKY_SFC_SW_DWN"]
for i in parametres:
    df3[i+"_liste"]=df3[i].apply(convertir_en_liste)
#print(df3.shape)

#... Pour pouvoir les explode par la suite
df_T2M=df3[['T2M_liste','latitude','longitude']].copy()
df_T2M=df_T2M.explode("T2M_liste")
df_T2M[['date','T2M']]=pd.DataFrame(df_T2M['T2M_liste'].tolist(),index=df_T2M.index)
df_T2M=df_T2M.drop(columns="T2M_liste")
df_T2M.to_csv("df_T2M.csv", index=False)


df_T2M_MAX=df3[['T2M_MAX_liste','latitude','longitude']].copy()
df_T2M_MAX=df_T2M_MAX.explode("T2M_MAX_liste")
df_T2M_MAX[['date','T2M_MAX']]=pd.DataFrame(df_T2M_MAX['T2M_MAX_liste'].tolist(),index=df_T2M_MAX.index)
df_T2M_MAX=df_T2M_MAX.drop(columns="T2M_MAX_liste")
df_T2M_MAX.to_csv("df_T2M_MAX.csv", index=False)


df_T2M_MIN=df3[['T2M_MIN_liste','latitude','longitude']].copy()
df_T2M_MIN=df_T2M_MIN.explode("T2M_MIN_liste")
df_T2M_MIN[['date','T2M_MIN']]=pd.DataFrame(df_T2M_MIN['T2M_MIN_liste'].tolist(),index=df_T2M_MIN.index)
df_T2M_MIN=df_T2M_MIN.drop(columns="T2M_MIN_liste")
df_T2M_MIN.to_csv("df_T2M_MIN.csv", index=False)

df_PRECTOTCORR=df3[['PRECTOTCORR_liste','latitude','longitude']].copy()
df_PRECTOTCORR=df_PRECTOTCORR.explode("PRECTOTCORR_liste")
df_PRECTOTCORR[['date','PRECTOTCORR']]=pd.DataFrame(df_PRECTOTCORR['PRECTOTCORR_liste'].tolist(),index=df_PRECTOTCORR.index)
df_PRECTOTCORR=df_PRECTOTCORR.drop(columns="PRECTOTCORR_liste")
df_PRECTOTCORR.to_csv("df_PRECTOTCORR.csv", index=False)

df_RH2M=df3[['RH2M_liste','latitude','longitude']].copy()
df_RH2M=df_RH2M.explode("RH2M_liste")
df_RH2M[['date','RH2M']]=pd.DataFrame(df_RH2M['RH2M_liste'].tolist(),index=df_RH2M.index)
df_RH2M=df_RH2M.drop(columns="RH2M_liste")
df_RH2M.to_csv("df_RH2M.csv", index=False)

df_WS2M=df3[['WS2M_liste','latitude','longitude']].copy()
df_WS2M=df_WS2M.explode("WS2M_liste")
df_WS2M[['date','WS2M']]=pd.DataFrame(df_WS2M['WS2M_liste'].tolist(),index=df_WS2M.index)
df_WS2M=df_WS2M.drop(columns="WS2M_liste")
df_WS2M.to_csv("df_WS2M.csv", index=False)

df_ALLSKY_SFC_SW_DWN=df3[['ALLSKY_SFC_SW_DWN_liste','latitude','longitude']].copy()
df_ALLSKY_SFC_SW_DWN=df_ALLSKY_SFC_SW_DWN.explode("ALLSKY_SFC_SW_DWN_liste")
df_ALLSKY_SFC_SW_DWN[['date','ALLSKY_SFC_SW_DWN']]=pd.DataFrame(df_ALLSKY_SFC_SW_DWN['ALLSKY_SFC_SW_DWN_liste'].tolist(),index=df_ALLSKY_SFC_SW_DWN.index)
df_ALLSKY_SFC_SW_DWN=df_ALLSKY_SFC_SW_DWN.drop(columns="ALLSKY_SFC_SW_DWN_liste")
df_ALLSKY_SFC_SW_DWN.to_csv("df_ALLSKY_SFC_SW_DWN.csv", index=False)



In [ ]:
df5=pd.merge(df_T2M, df_T2M_MAX, on=["date","latitude","longitude"], how='inner')
df5=pd.merge(df5, df_T2M_MIN, on=["date","latitude","longitude"], how='inner')
df5=pd.merge(df5, df_PRECTOTCORR, on=["date","latitude","longitude"], how='inner')
df5=pd.merge(df5, df_RH2M, on=["date","latitude","longitude"], how='inner')
df5=pd.merge(df5, df_WS2M, on=["date","latitude","longitude"], how='inner')
df5=pd.merge(df5, df_T2M_MAX, on=["date","latitude","longitude"], how='inner')
df5=pd.merge(df5, df_ALLSKY_SFC_SW_DWN, on=["date","latitude","longitude"], how='inner')
print(df5)
df5.shape

        latitude  longitude      date    T2M  T2M_MAX_x  T2M_MIN  PRECTOTCORR  \
0         33.014    -93.718  20150101   2.26       3.32     0.94        32.20   
1         33.014    -93.718  20150102   5.22       7.26     3.07         8.35   
2         33.014    -93.718  20150103   8.17      12.97     3.90        38.28   
3         33.014    -93.718  20150104   1.91       5.11    -4.03         0.02   
4         33.014    -93.718  20150105  -0.79       6.36    -5.61         0.00   
...          ...        ...       ...    ...        ...      ...          ...   
275683    33.019    -94.041  20250621  28.03      32.29    23.94         0.19   
275684    33.019    -94.041  20250622  28.13      33.10    23.47         2.62   
275685    33.019    -94.041  20250623  27.89      32.85    23.01         3.18   
275686    33.019    -94.041  20250624  27.73      32.42    23.04         2.90   
275687    33.019    -94.041  20250625  27.55      32.46    23.44         2.36   

         RH2M  WS2M  T2M_MA

(275688, 11)

In [ ]:
df5.to_csv("donnees_meteo_final.csv", index=False)